## XBRL US API - Full text search of reports
This sample Python code completes a full-text search of SEC reports in the XBRL US Public Filings Database and returns results from 2018 to the present.

**The document endpoint is a benefit of XBRL US Membership - join XBRL US for comprehensive access - https://xbrl.us/join.**
### Authenticate for access token
XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query.
Run the cell below, then type your XBRL US Web account email, account password, Client ID, and secret (get these from https://xbrl.us/access-token), pressing the Enter key on the keyboard after each entry.

In [ ]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode


class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    url = 'https://api.xbrl.us/oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

def refresh(info):
    refresh_auth = {
                'client_id': info.client_id,
				'client_secret' : info.client_secret,
				'grant_type' : 'refresh_token',
				'platform' : 'ipynb',
				'refresh_token' : info.refresh_token
                }
    refreshres = requests.post(info.url, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json['access_token']
    info.refresh_token = refresh_json['refresh_token']
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info

tokenInfo = tokenInfoClass()

tokenInfo.email = input('Enter your XBRL US Web account email: ')
tokenInfo.password = getpass.getpass(prompt='Password: ')
tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email,
            'client_id': tokenInfo.client_id,
            'client_secret' : tokenInfo.client_secret,
            'password' : tokenInfo.password,
            'grant_type' : 'password',
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.url, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s.  Run the first cell again and enter the credentials." % (auth_json['error_description']))
else:
    tokenInfo.access_token = auth_json['access_token']
    tokenInfo.refresh_token = auth_json['refresh_token']
    print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")

#print(vars(tokenInfo))
print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)

### Make a query
After the access token confirmation appears above, you can modify the query below, then use the **_Cell >> Run_** menu option from the cell **immediately below this text** to run the entire query for results.

The sample results are from 10+ years of data for companies in an SIC code, and may take several minutes to recreate. **The document endpoint is a benefit of XBRL US Membership** - see https://xbrl.us/membership to get started.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return.

In [ ]:
# Define the parameters of the query. This query returns all
# SEC reports where 'Microsoft' and 'internet of things'
# appear within 300 words of each other.

endpoint = 'document'

text_search = ["Microsoft NEAR/300 \"internet of things\""
                ]

report_source = ["SEC"
                ]

# Define data fields to return (multi-sort based on order)

fields = [ # this is the list of the characteristics of the data being returned by the query
         'entity.name.sort(ASC)',
         'dts.id.sort(DESC)',
         'document.type',
         'document.uri',
         'document.example'
         ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 10 # Set as '' to display all rows in the notebook

# Below is the list of what's being queried using the search endpoint.

params = {
         'document.text-search': ','.join(text_search),
         'report.source-name': ','.join(report_source),
         'fields': ','.join(fields)
         }

print('\n\nNext click the run button (Colab) or in the gray code cell below, then click the Run button above to execute the query for results.\n\n')

In [ ]:
# @title
### Execute the query with loop for all results
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        res_json = res.json()
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else:
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else:
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else:
        offset_value += res_json['paging']['limit']
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"

    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)

    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))

    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Mon Nov 18 16:20:57 2024 info@xbrl.us (client ID: 69e1257c ...) started the query and
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 258 records.

At Mon Nov 18 16:21:10 2024, the query finished with   258   rows returned in 0:00:12.859922 for 
https://api.xbrl.us/api/v1/document/search?unique&document.text-search=Microsoft+NEAR/300+"internet+of+things"&report.source-name=SEC&fields=entity.name.sort(ASC),dts.id.sort(DESC),document.type,document.uri,document.example


,entity.name,dts.id,document.type,document.uri,document.example
0,ABB LTD,183503,report,http://www.sec.gov/Archives/edgar/data/1091587/000110465917015892/a16-22210_620f.htm,... electric motors account for <b>nearly</b> two‑thirds of the ... connects motors with the <b>Internet</b> of <b>Things</b> (IoT). The Drives and ... -reaching strategic partnership with <b>Microsoft</b> to develop next‑generation ... to decrease by approximately $<b>300</b> million due to additional ...
1,ACCENTURE HOLDINGS PLC,267928,report,http://www.sec.gov/Archives/edgar/data/1647339/000164733917000019/acnholdings831201710k.htm,"... Amazon Web Services, Apple, Google, <b>Microsoft,</b> Oracle, Pegasystems, salesforce.com, SAP, ... intelligence, augmented reality, automation, blockchain, <b>Internet</b> of <b>Things,</b> quantum computing and as ... meet our needs in the <b>near</b> future. ITEM 3. LEGAL ..."
2,Accenture plc,347226,inline,http://www.sec.gov/Archives/edgar/data/1467373/000146737319000339/acn831201910k.htm,"... Web Services, Google, <b>Microsoft,</b> Oracle, Pegasystems, Salesforce, ... automation, blockchain, <b>Internet</b> of <b>Things,</b> quantum computing ... needs in the <b>near</b> future. ITEM 3 ... noncontrolling interests ( 159 ) <b>300</b> ( 2,075 ) Cash flow ..."
3,Accenture plc,304571,report,http://www.sec.gov/Archives/edgar/data/1467373/000146737318000318/acn831201810k.htm,"... Services, Apple, Google, <b>Microsoft,</b> Oracle, Pegasystems, Salesforce, ... automation, blockchain, <b>Internet</b> of <b>Things,</b> quantum computing ... needs in the <b>near</b> future. 22 Table ... attributable to noncontrolling interests <b>300</b> (2,075 ) (4,740 ) ? ..."
4,Accenture plc,267957,report,http://www.sec.gov/Archives/edgar/data/1467373/000146737317000430/acn831201710k.htm,"... Amazon Web Services, Apple, Google, <b>Microsoft,</b> Oracle, Pegasystems, salesforce.com, SAP, ... intelligence, augmented reality, automation, blockchain, <b>Internet</b> of <b>Things,</b> quantum computing and as ... meet our needs in the <b>near</b> future. ITEM 3. LEGAL ..."
...,...,...,...,...,...
253,UBI Blockchain Internet LTD-DE,265427,report,http://www.sec.gov/Archives/edgar/data/1500242/000149315217010221/forms-1a.htm,"... the blockchain technology and <b>internet</b> of <b>things</b> promote industrial information and ... research and development, including IBM, <b>Microsoft,</b> Intel, Blockstream and Thompson ... and advancement of blockchian, <b>internet</b> of <b>things,</b> and technological innovation platform ..."
254,UBI Blockchain Internet LTD-DE,257745,report,http://www.sec.gov/Archives/edgar/data/1500242/000149315217007856/form10-q.htm,"... of blockchain technology and <b>internet</b> of <b>things</b>. Blockchain technology-based applications ... and development, including IBM, <b>Microsoft,</b> Intel, Blockstream and Thompson ... and advancement of blockchain, <b>internet</b> of <b>things,</b> and technological innovation platform ..."
255,UBI Blockchain Internet LTD-DE,190593,report,http://www.sec.gov/Archives/edgar/data/1500242/000149315217007534/forms-1a.htm,"... the blockchain technology and <b>internet</b> of <b>things</b> promote industrial information and ... research and development, including IBM, <b>Microsoft,</b> Intel, Blockstream and Thompson ... and advancement of blockchian, <b>internet</b> of <b>things,</b> and technological innovation platform ..."
256,UBI Blockchain Internet LTD-DE,190488,report,http://www.sec.gov/Archives/edgar/data/1500242/000149315217007324/form10qa.htm,"... of blockchain technology and <b>internet</b> of <b>things</b>. Blockchain technology-based applications ... and development, including IBM, <b>Microsoft,</b> Intel, Blockstream and Thompson ... and advancement of blockchain, <b>internet</b> of <b>things,</b> and technological innovation platform ..."


In [ ]:
# If you run this program locally, you can save the output to a file on your computer (modify D:\results.csv to your system)
df.to_csv(r"D:\results.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('data.csv')
#!cp data.csv "drive/My Drive/"